# Arquitectura y concurrencia · notebook de las clases 11.2 y 11.3

Este notebook acompaña las páginas 9 a 12 de la unidad 11. Es el lugar
**verboso**: cada tema se explica antes de la celda, la celda imprime
mensajes claros y, después, una celda «Salida esperada» dice qué debe
aparecer. Las páginas traen sólo las celdas cortas; aquí se repite el
contexto para que puedas ejecutar cada tema sin volver a leer la página.

**Reglas de las celdas.**

- Se ejecutan **en orden**, de arriba abajo. Cada celda de código empieza con
  un rótulo `# celda N.M`; ese rótulo es el que citan las páginas.
- Usan `await` de nivel superior: el kernel de Jupyter ya tiene un event loop
  corriendo, así que `asyncio.run(...)` truena aquí y nunca aparece.
- Los valores de «Salida esperada» son **cualitativos o rangos**. Los PID, los
  nombres de hilo y los milisegundos exactos cambian en cada máquina y en
  cada corrida; lo que debe repetirse es la **forma** del resultado.
- Las celdas 11.1 y 12.1 lanzan su propio servidor (`app_arq.py`, un solo
  worker) en un subproceso y lo cierran en un `finally`. No necesitas abrir
  otra terminal.

## 0.1 · Arranque: un lanzador reutilizable para el servidor de la clase

El servidor de las demostraciones D1 y D2 es `app_arq.py`, una aplicación
FastAPI que vive junto a este notebook. Guarda su estado **en la memoria del
proceso** (el evento de D2, los contadores), por eso se lanza con **un solo
worker** y sin `--reload`: con dos workers cada uno tendría su propia
memoria y las mediciones no significarían lo que dicen.

La celda de abajo sólo **define** dos funciones; no lanza nada todavía:

- `lanza_servidor(puerto=None, plazo=20.0)` ejecuta
  `sys.executable -m uvicorn app_arq:app --port PUERTO --workers 1` en un
  subproceso (el `--workers 1` explícito evita que una variable de entorno
  como `WEB_CONCURRENCY` active varios y parta el estado en memoria),
  con la carpeta de este notebook como directorio de trabajo. Si no das
  puerto, pide uno libre al sistema operativo. Espera, con un plazo, a que
  `/quien` responda. Falla con un mensaje claro si el puerto ya está ocupado
  o si el proceso termina antes de responder (y muestra el final de su log).
  Devuelve el proceso, con la dirección en `proc.url`.
- `detiene_servidor(proc)` pide el cierre con `terminate`, espera hasta 5 s y,
  sólo si no cerró, lo mata con `kill`. Es seguro llamarla dos veces.

El patrón de uso, en todas las celdas que miden, es siempre el mismo:

```text
proc = lanza_servidor()
try:
    ...   # la medición
finally:
    detiene_servidor(proc)
```

Así el servidor se cierra aunque la medición falle o interrumpas el kernel.

In [ ]:
# celda 0.1
import os
import socket
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path

CARPETA = Path.cwd()   # Jupyter arranca el kernel en la carpeta del notebook


def puerto_libre():
    """Un puerto TCP que el sistema operativo declara libre ahora mismo."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def _ocupado(puerto):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", puerto)) == 0


def _final_del_log(proc, lineas=15):
    try:
        proc.log.flush()
        texto = Path(proc.log.name).read_text(errors="replace")
    except Exception:
        return "(sin log)"
    return "\n".join(texto.splitlines()[-lineas:]) or "(log vacío)"


def lanza_servidor(puerto=None, plazo=20.0):
    """Arranca app_arq con UN worker y espera a que /quien responda."""
    if not (CARPETA / "app_arq.py").is_file():
        raise FileNotFoundError(f"no encuentro app_arq.py en {CARPETA}")
    puerto = puerto or puerto_libre()
    if _ocupado(puerto):
        raise RuntimeError(f"el puerto {puerto} ya está ocupado: elige otro")
    log = tempfile.NamedTemporaryFile("w+", prefix="app_arq_", suffix=".log",
                                      delete=False)
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app_arq:app",
         "--port", str(puerto), "--workers", "1",
         "--log-level", "warning"],
        cwd=CARPETA, stdout=log, stderr=subprocess.STDOUT)
    proc.log = log
    proc.url = f"http://127.0.0.1:{puerto}"
    try:
        limite = time.monotonic() + plazo
        while time.monotonic() < limite:
            if proc.poll() is not None:
                raise RuntimeError(f"el servidor terminó antes de responder "
                                   f"(código {proc.returncode}). Final de su "
                                   f"log:\n{_final_del_log(proc)}")
            try:
                with urllib.request.urlopen(proc.url + "/quien", timeout=1) as r:
                    if r.status == 200:
                        return proc
            except OSError:
                time.sleep(0.1)
        raise TimeoutError(f"el servidor no respondió /quien en {plazo} s")
    except BaseException:       # también Ctrl-C / interrupción del kernel
        detiene_servidor(proc)  # cierra el hijo y borra el log
        raise


def detiene_servidor(proc):
    """terminate, espera acotada y, si hace falta, kill. Idempotente."""
    if proc is None:
        return
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    log = getattr(proc, "log", None)
    if log is not None and not log.closed:
        log.close()
        try:
            os.unlink(log.name)
        except OSError:
            pass


print("lanza_servidor y detiene_servidor definidas")
print("carpeta de trabajo:", CARPETA)
print("python del kernel :", sys.executable)

### Salida esperada de la celda 0.1

Tres renglones: el aviso de que las dos funciones quedaron definidas, la
carpeta donde está este notebook (debe contener `app_arq.py`) y la ruta del
Python del kernel (debe ser el de tu ambiente `uv`, no el del sistema). Si
la carpeta no es la del notebook, abre el notebook desde su carpeta o cambia
el directorio del kernel antes de seguir.

## 9.1 · Relojes locales: restar marcas no mide la latencia

**Qué vamos a ver.** Dos componentes, **A** y **B**, cada uno con su propio
reloj local. A envía un mensaje a B; cada uno anota la hora de **su** reloj:
A al enviar, B al recibir. Es una simulación con números enteros en
milisegundos: no usa el reloj real de tu máquina, y por eso la cuenta es
exacta.

**Las variables (todas en ms).**

- **`LATENCIA_MS` = 20:** lo que tarda el mensaje en llegar, en tiempo real
  (valor supuesto del ejemplo).
- **`DESFASE_B_MS` = −40:** el reloj de B marca 40 ms **menos** que el
  tiempo real (está atrasado). El de A no tiene desfase (0).
- **marca de un componente** = tiempo real + su desfase.

**La cuenta, paso a paso.**

1. **Envío.** A envía en el tiempo real **100**. Su marca es
   100 + 0 = **100**.
2. **Llegada.** El mensaje llega en el tiempo real
   100 + `LATENCIA_MS` = 100 + 20 = **120**.
3. **Marca de B.** B anota 120 + `DESFASE_B_MS` = 120 + (−40) = **80**.
4. **Resta de marcas.** marca de B − marca de A = 80 − 100 = **−20**.
5. **Por qué sale −20.** La resta mezcla dos relojes, así que es
   `LATENCIA_MS` + `DESFASE_B_MS` = 20 + (−40) = **−20**. La latencia real
   (20) queda tapada por el desfase (−40).

**Qué significa.**

- La resta es **negativa**: parece que el mensaje llegó antes de enviarse.
- No es un error del programa: es que **restar marcas de dos relojes
  distintos no mide la latencia**.
- Sin conocer el desfase (y su incertidumbre), la resta no dice cuánto tardó
  el mensaje.

In [ ]:
# celda 9.1
LATENCIA_MS = 20          # supuesto: lo que tarda el mensaje en tiempo real
DESFASE_B_MS = -40        # supuesto: el reloj de B marca 40 ms menos que el real


class Componente:
    """Un componente con reloj local = tiempo real + desfase."""

    def __init__(self, nombre, desfase_ms):
        self.nombre = nombre
        self.desfase_ms = desfase_ms

    def marca(self, real_ms):
        return real_ms + self.desfase_ms


a = Componente("A", 0)
b = Componente("B", DESFASE_B_MS)

envio_real = 100                       # A envía en el instante real 100 ms
llegada_real = envio_real + LATENCIA_MS

marca_a = a.marca(envio_real)          # lo que A anota
marca_b = b.marca(llegada_real)        # lo que B anota

print(f"A anota al enviar : {marca_a} ms  (reloj de A)")
print(f"B anota al recibir: {marca_b} ms  (reloj de B)")
print(f"resta de marcas   : {marca_b} - {marca_a} = {marca_b - marca_a} ms")
print(f"latencia real     : {llegada_real - envio_real} ms (la conocemos sólo porque simulamos)")
print(f"latencia + desfase: {LATENCIA_MS} + ({DESFASE_B_MS}) = {LATENCIA_MS + DESFASE_B_MS} ms")
assert (marca_a, marca_b, marca_b - marca_a) == (100, 80, -20)

### Salida esperada de la celda 9.1

Los valores son exactos porque es una simulación con enteros:

- A anota 100 ms y B anota 80 ms.
- La resta de marcas da **−20 ms** (negativa).
- La latencia real es 20 ms y `latencia + desfase` también da −20 ms.

**Lectura.** La diferencia entre marcas de dos máquinas no es la latencia. El
orden de eventos entre máquinas se reconstruye con **mensajes** («A envía,
luego B recibe»), no comparando relojes. Si al ejecutar la celda no ves el
−20, revisa que no hayas cambiado `DESFASE_B_MS` o `LATENCIA_MS`.

## 10.1 · Quién ejecuta: 4 hilos frente a 2 procesos

**Qué se va a ver.** El mismo trabajo, **4 tareas** de cálculo puro de
Python (cada una del orden de 100 ms, según tu CPU), se manda dos veces:

1. a un `ThreadPoolExecutor` de **4 hilos** (un solo proceso);
2. a un `ProcessPoolExecutor` de **2 procesos**, con un contexto de arranque
   explícito (`spawn`) para que se comporte igual en cualquier sistema.

Cada tarea devuelve **quién la ejecutó**: el PID del proceso y el nombre del
hilo. La función vive en `trabajo_arq.py` (junto a este notebook) porque el
proceso hijo la busca **importando** ese módulo; una función definida en una
celda no es importable por un hijo `spawn`.

Lo que la celda muestra es la **pertenencia**: con hilos, un PID y varios
nombres de hilo; con procesos, varios PID. La celda también imprime el
número de CPU lógicas de tu máquina, que es el tope de ejecución simultánea
del hardware. **No promete aceleración**: en CPython con GIL, 4 hilos de
cálculo puro no corren bytecode a la vez, y 2 procesos pagan el costo de
arrancar. La celda no compara tiempos.

In [ ]:
# celda 10.1
import multiprocessing
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import threading

from trabajo_arq import tarea

N_TAREAS = 4
cpus = getattr(os, "process_cpu_count", os.cpu_count)()
print(f"CPU lógicas disponibles para este proceso: {cpus}")

# 1. cuatro hilos, un proceso. La barrera hace que ninguna tarea empiece su
# cálculo hasta que las 4 estén dentro de un hilo distinto (si no, un hilo
# libre podría tomar dos tareas y verías 3 nombres de hilo)
barrera = threading.Barrier(N_TAREAS, timeout=30)


def tarea_con_barrera(i):
    barrera.wait()
    return tarea(i)


with ThreadPoolExecutor(max_workers=4, thread_name_prefix="hilo") as pool:
    con_hilos = list(pool.map(tarea_con_barrera, range(N_TAREAS)))

print("\n4 hilos:")
for r in con_hilos:
    print(f"  tarea {r['tarea']}: PID {r['pid']}  hilo {r['hilo']}")
pids_h = {r["pid"] for r in con_hilos}
hilos_h = {r["hilo"] for r in con_hilos}
print(f"  -> {len(pids_h)} PID y {len(hilos_h)} nombres de hilo distintos")

# 2. dos procesos (contexto explícito)
ctx = multiprocessing.get_context("spawn")
with ProcessPoolExecutor(max_workers=2, mp_context=ctx) as pool:
    con_procesos = list(pool.map(tarea, range(N_TAREAS)))

print("\n2 procesos:")
for r in con_procesos:
    print(f"  tarea {r['tarea']}: PID {r['pid']}  hilo {r['hilo']}")
pids_p = {r["pid"] for r in con_procesos}
print(f"  -> {len(pids_p)} PID distintos (y ninguno es el del kernel: {os.getpid()})")

assert len(pids_h) == 1 and len(hilos_h) == 4
assert 1 <= len(pids_p) <= 2 and os.getpid() not in pids_p

### Salida esperada de la celda 10.1

- Primero el número de CPU lógicas (un entero; en la máquina del profesor, 8).
- **4 hilos:** las cuatro tareas muestran **el mismo PID** (el del kernel) y
  cuatro nombres de hilo distintos (`hilo_0` a `hilo_3`). Cierra con
  «1 PID y 4 nombres de hilo distintos».
- **2 procesos:** las cuatro tareas muestran **PID distintos del kernel**:
  normalmente dos PID, cada uno repetido (cada proceso atendió dos tareas), y
  el nombre de hilo es `MainThread` en ambos (cada proceso hijo ejecuta en su
  hilo principal). Cierra con «2 PID distintos».

Los valores concretos de PID cambian en cada ejecución. Si ves un solo PID
en la segunda parte, un proceso atendió las cuatro tareas (es posible, no es
un error); repite la celda. **Lectura.** Hilos y procesos responden a
«quién ejecuta»; cuántos de ellos corren a la vez lo deciden además el
planificador del sistema operativo, las CPU y el GIL, no el tamaño del pool.

## 10.2 · ¿Cuántas CPU usa cada herramienta?

**Qué vamos a ver.** El **mismo cálculo** (Python puro, sin esperar nada),
repartido en **4 tareas**, se ejecuta con tres herramientas y se mide cuántas
CPU trabajaron a la vez en cada una:

1. **4 hilos** (`ThreadPoolExecutor(4)`): un proceso, cuatro hilos.
2. **4 procesos** (`ProcessPoolExecutor(4)` con contexto `spawn`): cuatro
   intérpretes de Python, cada uno con su propio GIL.
3. **`joblib.Parallel(n_jobs=4)`** con el backend por omisión (`loky`): una
   librería que reparte las tareas en procesos trabajadores.

**Qué medimos, con estas variables.**

- **reloj** (s): lo que pasó en tu reloj de pared, medido con
  `time.monotonic()`, desde antes de crear el pool hasta tener los resultados
  (incluye arrancar los procesos).
- **CPU** (s): el tiempo que las CPU estuvieron **ejecutando** nuestro
  código, sumado entre todos los hilos o procesos. Si dos CPU trabajan un
  segundo cada una, son 2 s de CPU en 1 s de reloj.
- **CPU efectivas** = CPU ÷ reloj: cuántas CPU, en promedio, estuvieron
  trabajando a la vez. Vale 1 si sólo una trabajó a la vez; vale 4 si cuatro
  trabajaron todo el tiempo.
- **PID distintos**: cuántos procesos distintos ejecutaron las tareas.

**Cómo se mide la CPU (y por qué no es la misma llamada en los tres).**

- **Hilos:** todos viven en **este** proceso, así que basta
  `time.process_time()` (CPU de este proceso, todos sus hilos).
- **Procesos:** `process_time()` **no** ve a los hijos (marcaría casi 0).
  Se usa `resource.getrusage(RUSAGE_CHILDREN)`, que suma los hijos que ya
  **terminaron y fueron esperados**; por eso se lee **después** de cerrar el
  pool. Incluye también el costo de arrancar cada proceso.
- **joblib (loky):** sus trabajadores **no terminan** al acabar las tareas
  (se quedan vivos para reutilizarse), así que `RUSAGE_CHILDREN` marca 0
  mientras sigan vivos. Aquí cada tarea mide su propia CPU con
  `time.thread_time()` (CPU del hilo que la ejecuta) y se suman. Al final la
  celda cierra a los trabajadores y muestra que entonces `RUSAGE_CHILDREN`
  sí los cuenta.

**Qué esperamos.**

- **Hilos: CPU efectivas cerca de 1.** Aunque haya 4 hilos, el **GIL** deja
  que sólo uno ejecute bytecode de Python a la vez. Los demás esperan el
  candado, y esperar no consume CPU.
- **Procesos: CPU efectivas claramente mayores que 1** (hasta 4, si tu
  máquina tiene al menos 4 CPU). Cada proceso tiene su propio GIL. Arrancar
  los procesos cuesta reloj, así que no llega al máximo.
- **joblib: parecido a los procesos**, porque usa procesos. Lo que hace
  `loky` es mantener **trabajadores reutilizables**: la segunda llamada
  reutiliza los mismos PID sin volver a arrancar procesos. La celda la hace
  dos veces para que lo veas.

In [ ]:
# celda 10.2
import multiprocessing
import os
import resource
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import joblib
from joblib.externals.loky import get_reusable_executor

from trabajo_arq import tarea_cpu

N_TAREAS = 4
print(f"os.cpu_count() = {os.cpu_count()}  (CPU lógicas de la máquina)")


def cpu_hijos():
    r = resource.getrusage(resource.RUSAGE_CHILDREN)
    return r.ru_utime + r.ru_stime


def mide(nombre, trabajo, cpu_de):
    """Corre `trabajo()` y devuelve (renglón, resultados). `cpu_de` dice cómo medir la CPU."""
    reloj0, cpu0, hijos0 = time.monotonic(), time.process_time(), cpu_hijos()
    resultados = trabajo()
    reloj = time.monotonic() - reloj0
    cpu = cpu_de(resultados, time.process_time() - cpu0, cpu_hijos() - hijos0)
    pids = {r["pid"] for r in resultados}
    return (nombre, reloj, cpu, cpu / reloj, len(pids)), resultados


def con_hilos():
    with ThreadPoolExecutor(max_workers=N_TAREAS) as pool:
        return list(pool.map(tarea_cpu, range(N_TAREAS)))


def con_procesos():
    ctx = multiprocessing.get_context("spawn")
    with ProcessPoolExecutor(max_workers=N_TAREAS, mp_context=ctx) as pool:
        return list(pool.map(tarea_cpu, range(N_TAREAS)))


def con_joblib():
    return joblib.Parallel(n_jobs=N_TAREAS)(
        joblib.delayed(tarea_cpu)(i) for i in range(N_TAREAS))


filas = []
f, _ = mide("4 hilos", con_hilos, lambda r, proc, hijos: proc)
filas.append(f)
f, res_proc = mide("4 procesos", con_procesos, lambda r, proc, hijos: hijos)
filas.append(f)
hijos_antes_jl = cpu_hijos()
f, res_jl1 = mide("joblib (1.a llamada)", con_joblib,
                  lambda r, proc, hijos: sum(x["cpu"] for x in r))
filas.append(f)
f, res_jl2 = mide("joblib (2.a llamada)", con_joblib,
                  lambda r, proc, hijos: sum(x["cpu"] for x in r))
filas.append(f)
hijos_con_loky_vivo = cpu_hijos() - hijos_antes_jl   # los trabajadores de loky siguen vivos

print(f"\n{'herramienta':<22} {'reloj (s)':>9} {'CPU (s)':>8} {'CPU efectivas':>14} {'PID distintos':>14}")
for nombre, reloj, cpu, efectivas, pids in filas:
    print(f"{nombre:<22} {reloj:>9.2f} {cpu:>8.2f} {efectivas:>14.2f} {pids:>14}")

pids_1 = {x["pid"] for x in res_jl1}
pids_2 = {x["pid"] for x in res_jl2}
print(f"\njoblib: PID de la 1.a llamada {sorted(pids_1)}")
print(f"        PID de la 2.a llamada {sorted(pids_2)}  -> mismos: {pids_1 == pids_2}")

# cerrar a los trabajadores de loky: sólo entonces RUSAGE_CHILDREN los cuenta
antes = cpu_hijos()
get_reusable_executor().shutdown(wait=True)
print(f"\nRUSAGE_CHILDREN durante las 2 llamadas de joblib (trabajadores vivos): +{hijos_con_loky_vivo:.2f} s")
print(f"RUSAGE_CHILDREN tras cerrar a los trabajadores de loky             : +{cpu_hijos() - antes:.2f} s")

efectivas = {f[0]: f[3] for f in filas}
assert hijos_con_loky_vivo < 0.05 and cpu_hijos() - antes > 0.5
assert efectivas["4 hilos"] < 1.5
assert len(pids_1) <= 4 and os.getpid() not in pids_1
if (os.cpu_count() or 1) >= 4:
    assert efectivas["4 procesos"] > 1.8 and efectivas["joblib (1.a llamada)"] > 1.5

### Salida esperada de la celda 10.2

Primero `os.cpu_count()` (en la máquina del profesor, 8). Después una tabla
con cuatro renglones. Los números cambian en cada máquina; lo que se
mantiene son estos rangos:

- **4 hilos:** 1 PID distinto y **CPU efectivas entre 0.8 y 1.3**. El GIL
  dejó trabajar a un hilo a la vez: la CPU total es casi igual al reloj.
- **4 procesos:** 4 PID distintos y **CPU efectivas entre 2 y 4** (con 4 o más
  CPU). Menos de 4 porque arrancar los procesos consume reloj.
- **joblib, 1.a llamada:** 4 PID distintos y **CPU efectivas entre 1.5 y 3**,
  algo menos que los procesos: `loky` arranca sus trabajadores e importa
  `joblib` en cada uno, y eso es reloj sin cálculo.
- **joblib, 2.a llamada:** los **mismos PID** que la primera, y un reloj
  **menor o igual**, porque ya no hay que arrancar procesos.
- Las dos últimas líneas: durante las dos llamadas de joblib
  `RUSAGE_CHILDREN` sube **+0.00 s** (los trabajadores siguen vivos, así que
  no cuentan), y tras cerrarlos sube varios segundos (en la máquina del
  profesor, de 7 a 9 s): todo el trabajo de las **dos** llamadas más el
  arranque de los 4 trabajadores, que se registra hasta que terminan.

Si tu máquina tiene menos de 4 CPU, «CPU efectivas» no pasa de ese número
(la celda omite entonces la comprobación de los procesos). **Lectura.** Los
hilos comparten el GIL y suman casi una CPU; los procesos (con
`ProcessPoolExecutor` o con `joblib`) tienen un GIL cada uno y suman varias.
`joblib` no es otro mecanismo: es una forma cómoda de usar procesos que
además los conserva entre llamadas.

## 11.1 · D1: la misma espera de 0.5 s, escrita de tres maneras

**Qué vamos a ver.** El servidor tiene tres endpoints que hacen lo mismo:
esperar `w = 0.5` segundos y responder. Lo único que cambia es **cómo**
esperan. Se mandan **n = 12 peticiones simultáneas** a cada uno, contra
**un solo worker** (un solo proceso), y se mide con `time.monotonic` el
**total**: desde que salen las 12 hasta que todas contestan.

**Qué esperamos.** Tres tiempos totales muy distintos para la misma espera.
Antes de correr la celda, estas son las tres cuentas; lo que cambia entre
ellas es **quién ejecuta la espera y si la espera bloquea un hilo**.

**Las variables (las mismas en las tres fórmulas).**

- **n**: número de peticiones simultáneas (aquí 12).
- **w**: segundos que espera cada petición (aquí 0.5).
- **k_h**: cuántos hilos tiene el pool donde FastAPI ejecuta las funciones
  `def` (el limitador de AnyIO; por omisión **40**).
- **T**: el tiempo total que mide la celda, de la primera petición enviada a
  la última respuesta recibida.

**Las tres versiones.**

1. **`/def-bloqueante`: `def` con `time.sleep(w)`.**
   - FastAPI manda cada petición a un **hilo del pool** (hasta k_h a la vez).
   - Cada hilo se duerme por su cuenta; un hilo dormido no impide que otro
     duerma.
   - Si n ≤ k_h, las n esperas corren a la vez. Si n > k_h, las que sobran
     esperan turno en tandas.
   - **T_def ≈ ⌈n / k_h⌉ · w.** Aquí ⌈12 / 40⌉ = 1 tanda, así que
     **T ≈ 0.5 s**.
2. **`/async-bloqueante`: `async def` con `time.sleep(w)`.**
   - Una `async def` corre en el hilo del **event loop**, que es **un solo
     hilo**.
   - `time.sleep` **bloquea** ese hilo: mientras duerme no se atiende
     ninguna otra petición ni se cuenta ningún tic.
   - Las n esperas se hacen una tras otra.
   - **T_async+sleep ≈ n · w.** Aquí 12 × 0.5 = **6 s**.
3. **`/async-await`: `async def` con `await asyncio.sleep(w)`.**
   - También corre en el hilo del loop, pero `await` **cede el hilo**: el
     loop deja esa petición esperando y atiende las otras 11.
   - Las n esperas se traslapan en un solo hilo.
   - **T_async+await ≈ w.** Aquí **T ≈ 0.5 s**.

**El latido.** Para ver el bloqueo, el servidor anota un tic cada 10 ms.

- Si el loop está libre hay un tic cada 10 ms (hueco ≈ 0.01 s).
- Si el loop está bloqueado, los tics se detienen.
- El **hueco máximo** entre tics es cuánto tiempo estuvo bloqueado el loop.
- La celda llama a `POST /reinicia` (pone el latido a cero) antes de cada
  versión y a `GET /latido` (devuelve `max_gap`) al terminar.

**Cómo se mide.** Antes de medir, la celda abre las 12 conexiones con una
petición a `/quien` cada una (calentamiento), para que el tiempo de
establecerlas no se mezcle con la espera. El servidor se lanza dentro de la
celda y se cierra en un `finally`; todo el bloque tiene un plazo de 60 s.

In [ ]:
# celda 11.1
import asyncio
import httpx

W = 0.5          # segundos que espera cada petición
N_PETICIONES = 12

LIMITES = httpx.Limits(max_connections=20, max_keepalive_connections=20)
PLAZOS = httpx.Timeout(30.0)       # lectura de 30 s: mayor que 2*12*w = 12 s

VERSIONES = (
    ("def + time.sleep", "cada petición va a su hilo", "/def-bloqueante"),
    ("async def + time.sleep", "un hilo, el sleep lo bloquea", "/async-bloqueante"),
    ("async def + await asyncio.sleep", "await cede el hilo", "/async-await"),
)


async def mide_version(base, ruta):
    """12 peticiones simultáneas a `ruta`; devuelve (total, hueco máximo del latido)."""
    async with httpx.AsyncClient(limits=LIMITES, timeout=PLAZOS) as cliente:
        # calienta las 12 conexiones (keep-alive) antes de medir
        await asyncio.gather(*(cliente.get(base + "/quien") for _ in range(N_PETICIONES)))
        await cliente.post(base + "/reinicia")          # latido a cero

        t0 = time.monotonic()
        respuestas = await asyncio.gather(*(
            cliente.get(f"{base}{ruta}?s={W}") for _ in range(N_PETICIONES)))
        total = time.monotonic() - t0
        hueco = (await cliente.get(base + "/latido")).json()["max_gap"]
    assert all(r.status_code == 200 for r in respuestas)
    return total, hueco


proc = lanza_servidor()
try:
    print(f"servidor en {proc.url} (un worker)")
    totales = {}
    async with asyncio.timeout(60):                      # plazo global
        for nombre, _, ruta in VERSIONES:
            print(f"midiendo {nombre} ...")
            totales[nombre] = await mide_version(proc.url, ruta)
finally:
    detiene_servidor(proc)

print(f"\n{N_PETICIONES} peticiones simultáneas, cada una espera {W} s")
print(f"{'versión':<33} {'qué pasa':<30} {'total (s)':>9} {'hueco máx. latido (s)':>22}")
for nombre, que_pasa, _ in VERSIONES:
    total, hueco = totales[nombre]
    print(f"{nombre:<33} {que_pasa:<30} {total:>9.2f} {hueco:>22.3f}")
print(f"esperado: total ~{W} s, ~{N_PETICIONES * W:.1f} s y ~{W} s; "
      f"hueco < 0.1 s, ~{N_PETICIONES * W:.1f} s y < 0.1 s")
print("\nLo que importa no es la palabra async, sino si la espera BLOQUEA el hilo "
      "o lo CEDE.")

(t_def, h_def), (t_blq, h_blq), (t_aw, h_aw) = (totales[n] for n, _, _ in VERSIONES)
assert t_blq >= 4 and h_blq >= 2, "la espera bloqueante debía tardar y bloquear el loop"
assert t_def <= 1.5 and t_aw <= 1.5, "las esperas que no bloquean el loop debían traslaparse"
assert h_def < 0.1 and h_aw < 0.1, "el loop debía seguir libre"

### Salida esperada de la celda 11.1

Una tabla con tres renglones, total y hueco máximo del latido **en rangos**,
no exactos:

- `def + time.sleep`: total **cerca de 0.5 s** (no más de 1.5 s) y hueco
  **menor que 0.1 s**. Las 12 esperas corrieron a la vez, cada una en su
  hilo, y el loop siguió libre.
- `async def + time.sleep`: total **cerca de 6 s** (al menos 4 s) y hueco
  **también cerca de 6 s** (al menos 2 s). Las 12 esperas se hicieron una
  tras otra porque `time.sleep` bloqueó el único hilo del loop, sin dejar
  correr los tics.
- `async def + await asyncio.sleep`: total **cerca de 0.5 s** (no más de
  1.5 s) y hueco **menor que 0.1 s**. El `await` cedió el hilo y las 12
  esperas se traslaparon.

Al final, una línea con la conclusión. Si el segundo renglón sale cerca de
0.5 s, revisa que el servidor tenga un solo worker. **Lectura.** Las
versiones 2 y 3 son las dos `async def`; la diferencia no está en esa palabra
sino en la línea de espera: `time.sleep` **bloquea** el hilo del loop,
`await asyncio.sleep` lo **cede**. Un `def` llamado desde dentro de un
`async def` no se desplaza a otro hilo: corre en el hilo del loop.

## 11.2 · Presupuesto de conexiones: R × N × k

**Qué vamos a ver.** Una cuenta, sin servidor: cuántas conexiones a la base
de datos puede llegar a abrir un despliegue, y si caben en lo que la base
admite.

**Las variables, una por una.**

- **R = 3**: réplicas, es decir, copias del servicio corriendo (por
  ejemplo, tres contenedores).
- **N = 4**: workers por réplica; cada worker es un **proceso** distinto.
- **k = 10**: conexiones **máximas** del pool de cada worker. Cada worker
  crea **su propio** pool; un pool no se comparte entre procesos.
- **max_connections = 100**: cuántas conexiones admite PostgreSQL en total
  (valor habitual; compruébalo con `SHOW max_connections;`).
- **plazas reservadas = 3**: conexiones que PostgreSQL guarda para
  superusuarios (`SHOW superuser_reserved_connections;`); los demás no
  pueden usarlas.

**La cuenta, paso a paso.**

1. **Workers en total** = R × N = 3 × 4 = **12** procesos con pool propio.
2. **Capacidad potencial** = R × N × k = 12 × 10 = **120** conexiones: es el
   máximo que los pools pueden llegar a abrir **a la vez**.
3. **Disponibles para la aplicación** = max_connections − plazas
   reservadas = 100 − 3 = **97**.
4. **Diferencia** = 120 − 97 = **23** conexiones de más.
5. **¿Cabe?** Sí sólo si 120 ≤ 97. No cabe.

**Supuestos.** 10 es el máximo **total** por pool y todas las conexiones van
contra el mismo servidor PostgreSQL.

**Qué esperamos.** La celda imprime esos mismos cinco números. Cambia R, N
o k y repite para ver cuándo la cuenta cabe.

In [ ]:
# celda 11.2
R, N, k = 3, 4, 10                    # réplicas, workers por réplica, conexiones por pool
max_connections = 100                 # valor habitual de PostgreSQL
reservadas = 3                        # plazas de superusuario

potencial = R * N * k
disponibles = max_connections - reservadas

print(f"capacidad potencial: R x N x k = {R} x {N} x {k} = {potencial} conexiones")
print(f"disponibles en la base: {max_connections} - {reservadas} = {disponibles}")
print(f"el máximo configurado excede lo disponible en {potencial - disponibles}")
print("¿cabe todo el máximo a la vez?", potencial <= disponibles)

### Salida esperada de la celda 11.2

Con los valores de partida (si los cambias, cambian las cifras):

- `120` conexiones de capacidad potencial.
- `97` disponibles.
- El máximo configurado excede lo disponible en **23**, y la respuesta a
  «¿cabe todo el máximo a la vez?» es `False`.

**Lectura.** 120 es capacidad **potencial**, no uso: mientras los pools no
se llenen, todo funciona. El problema aparece bajo carga, cuando varios
pools piden su máximo a la vez, y escalar réplicas multiplica esa presión
sobre la base. Cambia `R`, `N` o `k` y repite para ver cuándo la cuenta cabe
en 97: la celda no tiene aserciones, así que cualquier valor se imprime.

## 11.3 · Workers y cores: `uvicorn --workers 2` en vivo

**Qué vamos a ver.** Hasta ahora el servidor tenía **un** proceso. Aquí se
lanza con `--workers 2` y se miran cuatro cosas, en este orden:

1. **Procesos.** Cuántos procesos crea `uvicorn` y quién es hijo de quién
   (se lee con `ps`).
2. **PIDs.** Se hacen 20 peticiones a `/contador` y se cuenta cuántos PIDs
   distintos responden.
3. **Memoria.** `/contador` suma 1 a un contador que vive **en la memoria del
   proceso** que responde. Si cada PID lleva su propia cuenta, el estado en
   memoria **no se comparte** entre workers.
4. **Cálculo.** `/calcula?ms=300` es una función `def` que gasta 300 ms de
   **CPU** (no duerme: calcula). Se mandan **4 peticiones simultáneas**, con
   1 worker y luego con 2.

**Qué esperamos del cálculo.**

- **Variables.** **n = 4** peticiones, **c = 0.3 s** de CPU cada una,
  **P** = número de workers (procesos).
- **Con P = 1.** `def` manda cada petición a un hilo distinto, pero los
  hilos de un mismo proceso comparten el **GIL**: sólo uno ejecuta Python a
  la vez. Los cálculos se hacen uno tras otro: **T ≈ n · c = 1.2 s**.
- **Con P = 2.** Cada proceso tiene su propio intérprete y su propio GIL,
  así que dos cálculos corren a la vez en dos cores. Si las 4 peticiones se
  reparten 2 y 2: **T ≈ (n / P) · c = 0.6 s**.
- **Cuidado con el reparto.** Un worker se asigna a la **conexión**, no a
  cada petición: el sistema operativo le entrega a un worker cada conexión
  nueva y esa conexión se queda con él mientras siga abierta. Por eso la
  celda abre varias conexiones, mira qué PID atiende cada una y se queda con
  **dos por cada PID** antes de medir. Sin eso, 3 contra 1 es posible y el
  tiempo no mejora.

**Lanzamiento y limpieza.** La celda lanza su propio servidor (no usa
`lanza_servidor`, que fuerza un worker), en un puerto libre, y lo cierra en
un `finally`: manda `terminate` al proceso principal, espera, y si quedaron
procesos descendientes los mata. Al final verifica con `ps` que no quedó
ninguno.

In [ ]:
# celda 11.3
import asyncio
import collections
import httpx

N_CALCULOS, C_MS = 4, 300


def arbol(raiz):
    """Descendientes de `raiz` según `ps`: lista de (pid, ppid, comando)."""
    filas = subprocess.run(["ps", "-eo", "pid=,ppid=,args="], capture_output=True,
                           text=True).stdout.splitlines()
    todos = {}
    for fila in filas:
        pid, ppid, cmd = fila.split(None, 2)
        todos[int(pid)] = (int(ppid), cmd)
    desc, cola = [], [raiz]
    while cola:
        padre = cola.pop()
        for pid, (pp, cmd) in todos.items():
            if pp == padre:
                desc.append((pid, pp, cmd))
                cola.append(pid)
    return desc


def rol(cmd):
    """Nombre legible del proceso según su línea de comando."""
    if "resource_tracker" in cmd:
        return "resource_tracker (vigila recursos de multiprocessing)"
    if "spawn_main" in cmd:
        return "worker (lanzado con multiprocessing, modo spawn)"
    return cmd[:60]


def lanza_con_workers(n_workers, puerto, plazo=25.0):
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app_arq:app", "--port", str(puerto),
         "--workers", str(n_workers), "--log-level", "warning"],
        cwd=CARPETA, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    url = f"http://127.0.0.1:{puerto}"
    limite = time.monotonic() + plazo
    while time.monotonic() < limite:
        if proc.poll() is not None:
            raise RuntimeError(f"el servidor terminó (código {proc.returncode})")
        try:
            with urllib.request.urlopen(url + "/quien", timeout=1) as r:
                if r.status == 200:
                    return proc, url
        except OSError:
            time.sleep(0.1)
    raise TimeoutError(f"sin respuesta de /quien en {plazo} s")


def cierra_con_workers(proc):
    """terminate al principal; mata lo que siga vivo de sus descendientes."""
    if proc is None:
        return
    restos = [pid for pid, _, _ in arbol(proc.pid)]
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=8)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    time.sleep(0.5)
    for pid in restos:
        try:
            os.kill(pid, 9)       # si el pid ya no existe, no hay nada que matar
        except ProcessLookupError:
            pass


async def conexiones_repartidas(url, por_pid, maximo=24):
    """Abre conexiones una a una y conserva `por_pid` por cada PID distinto."""
    elegidas, cerrar = collections.defaultdict(list), []
    for _ in range(maximo):
        c = httpx.AsyncClient(limits=httpx.Limits(max_connections=1), timeout=30)
        pid = (await c.get(url + "/quien")).json()["pid"]
        if len(elegidas[pid]) < por_pid:
            elegidas[pid].append(c)
        else:
            cerrar.append(c)
        if len(elegidas) >= 2 and all(len(v) >= por_pid for v in elegidas.values()):
            break
    for c in cerrar:
        await c.aclose()
    return [c for v in elegidas.values() for c in v]


async def mide_calculo(url, n_workers):
    if n_workers == 1:
        clientes = [httpx.AsyncClient(timeout=30) for _ in range(N_CALCULOS)]
        for c in clientes:
            await c.get(url + "/quien")                 # abre la conexión
    else:
        clientes = await conexiones_repartidas(url, N_CALCULOS // n_workers)
    try:
        t0 = time.monotonic()
        rs = await asyncio.gather(*(c.get(f"{url}/calcula?ms={C_MS}") for c in clientes))
        return time.monotonic() - t0, collections.Counter(r.json()["pid"] for r in rs)
    finally:
        for c in clientes:
            await c.aclose()


puerto = puerto_libre()
proc = None
try:
    # ---- 1) dos workers: procesos, PIDs, contadores ----
    proc, url = lanza_con_workers(2, puerto)
    print(f"proceso principal (supervisor): PID {proc.pid}")
    print("descendientes según ps:")
    for pid, ppid, cmd in arbol(proc.pid):
        print(f"  PID {pid}  padre {ppid}  {rol(cmd)}")

    ultimos, veces = {}, collections.Counter()
    async with httpx.AsyncClient(timeout=10) as c:       # una sola conexión
        for _ in range(20):
            r = (await c.get(url + "/contador")).json()
            ultimos[r["pid"]] = r["contador"]
            veces[r["pid"]] += 1
    print(f"\n20 peticiones a /contador con UNA conexión: {len(veces)} PID distinto(s)")
    for pid in veces:
        print(f"  PID {pid}: respondió {veces[pid]} veces, su contador va en {ultimos[pid]}")
    veces2, ultimos2 = collections.Counter(), {}
    for _ in range(20):                                   # una conexión nueva cada vez
        r = httpx.get(url + "/contador").json()
        ultimos2[r["pid"]] = r["contador"]
        veces2[r["pid"]] += 1
    print(f"20 peticiones más, cada una con conexión nueva: {len(veces2)} PID distinto(s)")
    for pid in veces2:
        print(f"  PID {pid}: respondió {veces2[pid]} veces, su contador va en {ultimos2[pid]}")

    # ---- 2) cálculo con 2 workers ----
    async with asyncio.timeout(60):
        t2, quienes2 = await mide_calculo(url, 2)
    cierra_con_workers(proc)
    proc = None

    # ---- 3) cálculo con 1 worker ----
    proc, url = lanza_con_workers(1, puerto_libre())
    async with asyncio.timeout(60):
        t1, quienes1 = await mide_calculo(url, 1)
finally:
    cierra_con_workers(proc)

print(f"\n{N_CALCULOS} cálculos simultáneos de {C_MS} ms de CPU cada uno")
print(f"{'workers':>8} {'PIDs que atendieron':>20} {'total (s)':>10} {'esperado':>12}")
print(f"{1:>8} {len(quienes1):>20} {t1:>10.2f} {'~' + str(N_CALCULOS * C_MS / 1000) + ' s':>12}")
print(f"{2:>8} {len(quienes2):>20} {t2:>10.2f} {'~' + str(N_CALCULOS * C_MS / 2000) + ' s':>12}")
print(f"\nprocesos que quedan del servidor: "
      f"{len([1 for _, _, cmd in arbol(os.getpid()) if 'app_arq' in cmd or 'multiprocessing' in cmd])}")

### Salida esperada de la celda 11.3

**Procesos (lo que se observó con uvicorn 0.54).** El proceso que lanzaste
es el **supervisor**; `ps` lo muestra con la línea `python -m uvicorn
app_arq:app ... --workers 2`. Sus hijos son **3**: los **2 workers**, que
`ps` muestra como `python -c "from multiprocessing.spawn import spawn_main
..."` (no dicen `uvicorn` ni `app_arq`) y un `resource_tracker` de
`multiprocessing`. Por eso `ps | grep uvicorn` ve **un** solo proceso aunque
haya dos workers: para contarlos hay que mirar los **hijos** del supervisor.
El supervisor no atiende peticiones: crea el socket de escucha, lanza los
workers y los vigila.

**PIDs y contadores.**

- Con **una** conexión abierta, las 20 peticiones las atiende **el mismo
  PID** y su contador llega a 20.
- Con una conexión **nueva** por petición, responden **2 PIDs distintos**
  (el reparto no es exactamente 10 y 10) y **cada PID lleva su propio
  contador**: los contadores no suman 20 juntos, cada uno cuenta sólo lo
  suyo. El estado en memoria **no se comparte** entre workers.
- **Cómo se reparten.** El supervisor crea **un** socket de escucha y los
  workers lo heredan y hacen `accept` sobre él (no usan `SO_REUSEPORT`); el
  sistema operativo entrega cada conexión nueva a un worker que la espere.
  El reparto depende del sistema operativo, no es un turno estricto.

**Cálculo (rangos).**

- **1 worker:** total **entre 1.0 y 1.8 s** (≈ 4 × 0.3): el GIL serializa
  los cálculos aunque haya cuatro hilos.
- **2 workers:** total **entre 0.5 y 1.2 s** (cerca de la mitad) y **2 PIDs**
  distintos atendiendo.
- Al final debe decir **0** procesos del servidor.

**Lectura.** Más workers = más procesos = más GIL independientes = más
cores trabajando a la vez en cálculo. A cambio, cada worker tiene **su**
memoria (y su propio pool de conexiones, como en 11.2): lo que debe
compartirse tiene que vivir fuera del proceso (base de datos, Redis).

## 11.4 · El pool de Postgres visto desde el servidor

**Qué se va a ver.** En la celda 11.2 se hizo una cuenta, `R × N × k`, de
cuántas conexiones **podrían** llegar a la base. Aquí se mira una base
PostgreSQL real, por dentro, para ver qué es exactamente cada conexión y qué
hace un pool. La celda arranca un contenedor **temporal** de Docker con
PostgreSQL, lo usa y lo borra.

**Variables (definiciones, una por una).**

- **Conexión**: un canal abierto entre tu programa y la base, por el que
  viajan las consultas. En el programa es un objeto `psycopg.Connection`.
- **Backend**: el **proceso del sistema operativo**, del lado de PostgreSQL,
  que atiende una conexión. PostgreSQL crea **un backend por cada conexión**
  que se abre, y ese proceso vive en la máquina de la **base**, no en la de
  tu servidor web.
- **PID**: el número con que el sistema operativo identifica un proceso. Si
  dos conexiones tienen PID distinto, son dos procesos distintos.
- **Consulta activa**: una consulta que **se está ejecutando ahora**. En
  `pg_stat_activity` aparece como `state = 'active'`.
- **Conexión ociosa**: una conexión abierta que **no está ejecutando nada**
  (`state = 'idle'`). Sigue ocupando su backend y una plaza de la base.
- **Pool**: un conjunto de conexiones ya abiertas que el programa **presta**
  y devuelve, en vez de abrir y cerrar una por cada consulta.
- **Tamaño del pool**: cuántas conexiones puede tener. En `psycopg_pool` son
  dos números: `min_size` (las que abre al inicio y mantiene) y `max_size`
  (el tope; el pool crece hasta ahí sólo si hay quien espere).

**Qué esperamos (cuatro observaciones, A a D).**

1. **A.** Un pool de 10 conexiones abiertas son **10 backends** con **PID
   distinto**, y esos PID son procesos del contenedor (de la base), no del
   notebook.
2. **B.** Una conexión abierta no es una consulta activa: con 3 consultas de
   `pg_sleep(2)` corriendo a la vez, 3 conexiones están `active` y las otras
   7 siguen `idle`.
3. **C.** El pool es un **tope**. Con un pool de **2** conexiones y **6**
   tareas de 0.5 s, sólo 2 corren a la vez; las demás **esperan** una
   conexión (igual que en la página 5 esperaba el hilo). Cuenta:
   1. tareas por tanda = tamaño del pool = 2;
   2. tandas = ⌈6 / 2⌉ = 3;
   3. tiempo ≈ 3 × 0.5 s = **1.5 s**.
4. **D.** La base tiene su propio tope, `max_connections`, y reserva unas
   plazas (`superuser_reserved_connections`) para administradores. Se leen
   con `SHOW` y se comparan con la cuenta de la celda 11.2.

**Cómo funciona la celda.** Requiere Docker en marcha.

1. Lanza `postgres:17-alpine` con nombre único `fdd-pg-<aleatorio>`,
   contraseña de prueba y un puerto libre del equipo (sólo `127.0.0.1`).
2. Espera, con plazo de 60 s, a que la base acepte conexiones.
3. Hace las observaciones A a D.
4. En el `finally` borra el contenedor con `docker rm -f`. No toca ningún
   otro contenedor.

Si Docker no está disponible o el arranque falla, la celda lo dice y
muestra una **captura de salida previa** (rotulada «no es una ejecución»).

In [ ]:
# celda 11.4
import concurrent.futures as cf
import secrets
import socket
import subprocess
import time

CAPTURA_PREVIA = """\
contenedor fdd-pg-9fa5fd75 (postgres:17-alpine), servidor PostgreSQL 17.11, puerto 57203

== A. Un pool de 10 conexiones = 10 backends ==
conexiones del pool vistas en pg_stat_activity: 10
PID distintos: 10  ->  [61, 62, 63, 64, 65, 66, 67, 68, 69, 70]
de esos PID, procesos que existen DENTRO del contenedor: 10 de 10
tres procesos backend según `ps` dentro del contenedor (el primero puede ser la conexión de monitoreo):
    60 postgres: postgres postgres 172.17.0.1(49098) idle
    61 postgres: postgres postgres 172.17.0.1(49104) idle
    62 postgres: postgres postgres 172.17.0.1(49108) idle

== B. Conexión ociosa (idle) vs consulta activa (active) ==
con 3 `SELECT pg_sleep(2)` en curso: 10 conexiones abiertas, 3 active, 7 idle
2 s después, sin consultas: 10 conexiones abiertas, 0 active
tras cerrar el pool: 0 conexiones del pool en la base

== C. El pool es un tope: pool de 2, 6 tareas de 0.5 s ==
tamaño del pool = 2, tareas = 6, cada una pg_sleep(0.5)
tiempo total: 1.51 s   (cuenta: ceil(6/2) x 0.5 = 1.50 s)
duración de cada tarea (incluye esperar conexión): 0.50, 0.50, 1.01, 1.01, 1.51, 1.51
backends del pool2 en la base: 2 (nunca más de 2)

== D. Los topes de la base ==
max_connections = 100
superuser_reserved_connections = 3
plazas para clientes normales = 100 - 3 = 97
cuenta de la celda 11.2: R x N x k = 3 x 4 x 10 = 120  ->  NO cabe

contenedor fdd-pg-9fa5fd75 borrado"""


def docker(*args, plazo=60):
    return subprocess.run(["docker", *args], capture_output=True, text=True,
                          timeout=plazo, stdin=subprocess.DEVNULL)


def puerto_libre():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def espera_postgres(dsn, plazo=60.0):
    """Reintenta conectar hasta `plazo` s; devuelve la versión del servidor."""
    import psycopg
    fin = time.monotonic() + plazo
    ultimo = None
    while time.monotonic() < fin:
        try:
            with psycopg.connect(dsn, connect_timeout=2) as c:
                return c.execute("SHOW server_version").fetchone()[0]
        except psycopg.OperationalError as e:
            ultimo = e
            time.sleep(0.5)
    raise TimeoutError(f"Postgres no aceptó conexiones en {plazo:.0f} s: {ultimo}")


def estados(monitor, app):
    """[(pid, state)] de las conexiones cuyo application_name es `app`."""
    return monitor.execute(
        "SELECT pid, state FROM pg_stat_activity "
        "WHERE backend_type = 'client backend' AND application_name = %s ORDER BY pid",
        (app,)).fetchall()


def corre_pg():
    import psycopg
    from psycopg_pool import ConnectionPool

    nombre = f"fdd-pg-{secrets.token_hex(4)}"
    puerto = puerto_libre()
    dsn = f"host=127.0.0.1 port={puerto} user=postgres password=prueba dbname=postgres"
    creado = False
    try:
        r = docker("run", "-d", "--name", nombre, "-e", "POSTGRES_PASSWORD=prueba",
                   "-p", f"127.0.0.1:{puerto}:5432", "postgres:17-alpine", plazo=120)
        if r.returncode != 0:
            raise RuntimeError(f"docker run falló: {r.stderr.strip()[:300]}")
        creado = True
        version = espera_postgres(dsn)
        print(f"contenedor {nombre} (postgres:17-alpine), servidor PostgreSQL {version}, "
              f"puerto {puerto}\n")

        with psycopg.connect(dsn, autocommit=True,
                             application_name="fdd-monitor") as monitor:
            # ---------- A ----------
            print("== A. Un pool de 10 conexiones = 10 backends ==")
            with ConnectionPool(dsn, min_size=10, max_size=10, open=True,
                                kwargs={"application_name": "fdd-pool",
                                        "autocommit": True}) as pool:
                pool.wait(timeout=30)                  # espera a que abra las 10
                filas = estados(monitor, "fdd-pool")
                pids = {p for p, _ in filas}
                print(f"conexiones del pool vistas en pg_stat_activity: {len(filas)}")
                print(f"PID distintos: {len(pids)}  ->  {sorted(pids)}")
                ps = docker("exec", nombre, "ps", "-o", "pid,args").stdout
                en_contenedor = {int(l.split()[0]) for l in ps.splitlines()[1:]
                                 if l.split() and l.split()[0].isdigit()}
                print(f"de esos PID, procesos que existen DENTRO del contenedor: "
                      f"{len(pids & en_contenedor)} de {len(pids)}")
                muestra = [l.strip() for l in ps.splitlines() if "postgres: postgres" in l][:3]
                print("tres procesos backend según `ps` dentro del contenedor (el primero puede ser la conexión de monitoreo):")
                for l in muestra:
                    print("   ", l[:90])

                # ---------- B ----------
                print("\n== B. Conexión ociosa (idle) vs consulta activa (active) ==")

                def duerme2(_):
                    with pool.connection() as c:
                        c.execute("SELECT pg_sleep(2)")

                with cf.ThreadPoolExecutor(3) as ex:
                    futuros = [ex.submit(duerme2, i) for i in range(3)]
                    time.sleep(0.7)                    # a mitad de las consultas
                    en_vuelo = estados(monitor, "fdd-pool")
                    for f in futuros:
                        f.result()
                activas = sum(1 for _, s in en_vuelo if s == "active")
                ociosas = sum(1 for _, s in en_vuelo if s == "idle")
                print(f"con 3 `SELECT pg_sleep(2)` en curso: {len(en_vuelo)} conexiones "
                      f"abiertas, {activas} active, {ociosas} idle")
                despues = estados(monitor, "fdd-pool")
                print(f"2 s después, sin consultas: {len(despues)} conexiones abiertas, "
                      f"{sum(1 for _, s in despues if s == 'active')} active")

            # el pool se cerró: sus backends desaparecen
            time.sleep(0.5)
            print(f"tras cerrar el pool: {len(estados(monitor, 'fdd-pool'))} conexiones "
                  f"del pool en la base")

            # ---------- C ----------
            print("\n== C. El pool es un tope: pool de 2, 6 tareas de 0.5 s ==")
            with ConnectionPool(dsn, min_size=2, max_size=2, open=True,
                                kwargs={"application_name": "fdd-pool2",
                                        "autocommit": True}) as pool2:
                pool2.wait(timeout=30)

                def tarea(_):
                    t = time.monotonic()
                    with pool2.connection() as c:
                        c.execute("SELECT pg_sleep(0.5)")
                    return time.monotonic() - t

                t0 = time.monotonic()
                with cf.ThreadPoolExecutor(6) as ex:
                    espera = sorted(ex.map(tarea, range(6)))
                total = time.monotonic() - t0
                print(f"tamaño del pool = 2, tareas = 6, cada una pg_sleep(0.5)")
                print(f"tiempo total: {total:.2f} s   (cuenta: ceil(6/2) x 0.5 = 1.50 s)")
                print("duración de cada tarea (incluye esperar conexión): "
                      + ", ".join(f"{t:.2f}" for t in espera))
                print(f"backends del pool2 en la base: {len(estados(monitor, 'fdd-pool2'))} "
                      "(nunca más de 2)")

            # ---------- D ----------
            print("\n== D. Los topes de la base ==")
            maxc = int(monitor.execute("SHOW max_connections").fetchone()[0])
            res = int(monitor.execute(
                "SHOW superuser_reserved_connections").fetchone()[0])
            print(f"max_connections = {maxc}")
            print(f"superuser_reserved_connections = {res}")
            print(f"plazas para clientes normales = {maxc} - {res} = {maxc - res}")
            potencial = 3 * 4 * 10
            print(f"cuenta de la celda 11.2: R x N x k = 3 x 4 x 10 = {potencial}"
                  f"  ->  {'cabe' if potencial <= maxc - res else 'NO cabe'}")
    finally:
        if creado:
            docker("rm", "-f", nombre, plazo=60)
            print(f"\ncontenedor {nombre} borrado")


try:
    ok = docker("info", "--format", "{{.ServerVersion}}", plazo=15)
    if ok.returncode != 0:
        raise RuntimeError(ok.stderr.strip()[:200] or "docker info falló")
    corre_pg()
except Exception as e:                  # Docker ausente, caído o arranque fallido
    print(f"No se pudo correr la celda con Docker real: {type(e).__name__}: {e}")
    print("\n--- captura de salida previa (no es una ejecución) ---")
    print(CAPTURA_PREVIA)

### Salida esperada de la celda 11.4

Los PID, el nombre del contenedor y los tiempos cambian en cada corrida; lo
que debe repetirse es la **forma**:

- **Encabezado**: nombre `fdd-pg-…` y la versión de PostgreSQL (17.x).
- **A.** `10` conexiones, `10` PID distintos, y «10 de 10» PID que existen
  dentro del contenedor. Cada conexión del pool es un proceso `postgres:`
  aparte, en la base.
- **B.** Con las 3 consultas en curso: `10` conexiones abiertas, **3
  `active` y 7 `idle`**. Sin consultas, 0 activas. Tras cerrar el pool, 0
  conexiones: cerrar el pool cierra los backends.
- **C.** Total **cerca de 1.5 s** (entre 1.4 y 2 s); las duraciones
  individuales salen en tres grupos (≈0.5, ≈1.0, ≈1.5 s): las que tuvieron
  que esperar una conexión. Nunca más de 2 backends.
- **D.** `max_connections = 100`, `superuser_reserved_connections = 3`,
  97 plazas; la cuenta `3 × 4 × 10 = 120` **no cabe**.
- Última línea: «contenedor … borrado». Comprueba en una terminal que no
  quedó nada: `docker ps -a | grep fdd-pg` no debe imprimir nada.

**Lectura.** Una conexión **no** es gratis: es un proceso en la base, aunque
esté ociosa. Un pool reparte pocas conexiones entre muchas tareas, y por eso
sirve de tope (las tareas esperan), pero cada **worker** de cada **réplica**
tiene el suyo, y la base sólo tiene `max_connections` plazas.

## 12.1 · D2: cómo llegan los resultados (polling, long polling y SSE)

**Qué vamos a ver.** Un evento que ocurre en el servidor **10 s** después de
`POST /inicia?t=10`. Hay tres maneras de que los clientes se enteren; se
prueba cada una **por separado**, con **20 clientes**, reiniciando el
servidor y volviendo a iniciar el evento.

**Las tres modalidades.**

- **Polling:** cada cliente pregunta a `/listo` cada Δ segundos.
- **Long polling:** cada cliente hace **una** petición a `/espera` y el
  servidor la mantiene abierta hasta que ocurre el evento.
- **SSE:** cada cliente abre **una** respuesta HTTP continua en `/eventos`
  (streaming); el servidor envía latidos y, al final, el evento `listo`.

**Las variables del polling.**

- **T = 10 s:** cuándo ocurre el evento.
- **Δ = 3 s:** cada cuánto pregunta un cliente. Mide con reloj monotónico y
  no hace consulta inicial: pregunta en los segundos 3, 6, 9 y 12.
- **Clientes = 20.**

**La cuenta esperada del polling, paso a paso.**

1. **Consultas por cliente.** Las que hace hasta enterarse:
   ⌈T/Δ⌉ = ⌈10/3⌉ = 4 (en los segundos 3, 6, 9 y 12).
2. **Consultas sin novedades por cliente.** Las que llegan antes del
   evento: 4 − 1 = **3** (las de los segundos 3, 6 y 9).
3. **Demora por cliente.** Desde el evento hasta la consulta que lo
   descubre: Δ·⌈T/Δ⌉ − T = 3·4 − 10 = **2 s**.
4. **Con 20 clientes.** Consultas: 20 × 4 = **80**. Sin novedades:
   20 × 3 = **60**.

**Qué esperamos de las otras dos.** Long polling y SSE no preguntan una y
otra vez: **20 peticiones** en total (una por cliente) y una demora **casi
cero**, porque el servidor avisa en cuanto ocurre el evento.

**Qué se mide, por modalidad.** Son cosas distintas, no las confundas:

- **consultas:** las peticiones que recibió el servidor.
- **demora de detección:** desde el instante del evento hasta que el
  cliente se entera (promedio de los 20).
- **long polls pendientes:** peticiones a `/espera` que el servidor retiene
  abiertas a mitad de la espera (sólo cuenta esas).
- **sesiones SSE abiertas:** las que reporta `/estado` a mitad de la espera.
- **sockets del cliente:** conexiones que conserva el cliente (el polling
  con keep-alive también las conserva entre consultas, aunque no tenga
  peticiones pendientes).

**Duración.** Cada modalidad dura ~10-12 s, así que la celda tarda cerca de
35 s. Todo el bloque tiene un plazo global y el servidor se cierra en un
`finally`.

In [ ]:
# celda 12.1
import asyncio
import httpx

CLIENTES = 20
T_EVENTO = 10.0     # segundos hasta el evento
DELTA = 3.0         # periodo del polling
PLAZOS = httpx.Timeout(30.0)
LIMITES = httpx.Limits(max_connections=CLIENTES + 5,
                       max_keepalive_connections=CLIENTES + 5)


def sockets_de(cliente):
    """Conexiones que el cliente httpx tiene abiertas; None si no se puede inspeccionar."""
    try:
        return len(cliente._transport._pool.connections)
    except AttributeError:
        return None


async def polling(base, t_evento, cliente):
    inicio = t_evento - T_EVENTO
    consultas, k = 0, 1
    while True:
        await asyncio.sleep(max(0.0, inicio + k * DELTA - time.monotonic()))
        r = await cliente.get(base + "/listo")
        consultas += 1
        if r.json()["listo"]:
            return time.monotonic() - t_evento, consultas
        k += 1


async def long_polling(base, t_evento, cliente):
    r = await cliente.get(base + "/espera")
    assert r.json()["listo"]
    return time.monotonic() - t_evento, 1


async def sse(base, t_evento, cliente):
    async with cliente.stream("GET", base + "/eventos") as r:
        async for linea in r.aiter_lines():
            if linea.startswith("event: listo"):
                return time.monotonic() - t_evento, 1
    raise RuntimeError("el flujo SSE terminó sin el evento listo")


async def corre_modalidad(base, funcion):
    """Una modalidad completa: reinicia, inicia el evento, 20 clientes, estado."""
    async with httpx.AsyncClient(timeout=PLAZOS) as control:
        await control.post(base + "/reinicia")
        # /inicia devuelve el instante monotónico en que el SERVIDOR fijó el
        # evento; la demora se mide contra ese instante (mismo reloj: el
        # cliente y el servidor corren en esta máquina)
        t_evento = (await control.post(base + f"/inicia?t={T_EVENTO}")).json()["evento"]
        clientes = [httpx.AsyncClient(limits=LIMITES, timeout=PLAZOS)
                    for _ in range(CLIENTES)]
        try:
            tareas = [asyncio.create_task(funcion(base, t_evento, c)) for c in clientes]
            await asyncio.sleep(T_EVENTO / 2)            # a mitad de la espera
            medio = (await control.get(base + "/estado")).json()
            cuentas = [sockets_de(c) for c in clientes]
            sockets = None if None in cuentas else sum(cuentas)
            resultados = await asyncio.gather(*tareas)
            await asyncio.sleep(0.5)                     # deja cerrar los flujos
            final = (await control.get(base + "/estado")).json()
        finally:
            for t in locals().get("tareas", []):
                t.cancel()
            for c in clientes:
                await c.aclose()
    demoras = [d for d, _ in resultados]
    return {"consultas": final["consultas"] if funcion is polling
            else sum(n for _, n in resultados),
            "demora": sum(demoras) / len(demoras),
            "pendientes": medio["esperando"], "sesiones_sse": medio["sse"],
            "sockets": sockets, "sse_al_final": final["sse"]}


MODALIDADES = (("polling", polling), ("long polling", long_polling), ("SSE", sse))

proc = lanza_servidor()
try:
    print(f"servidor en {proc.url} (un worker)")
    medidas = {}
    async with asyncio.timeout(60):                      # plazo global
        for nombre, funcion in MODALIDADES:
            print(f"corriendo {nombre} con {CLIENTES} clientes (~{T_EVENTO:.0f} s) ...")
            medidas[nombre] = await corre_modalidad(proc.url, funcion)
finally:
    detiene_servidor(proc)

print(f"\nevento en T = {T_EVENTO:.0f} s, {CLIENTES} clientes por modalidad")
print(f"{'modalidad':<13} {'consultas':>9} {'demora (s)':>10} "
      f"{'long polls pendientes':>21} {'sesiones SSE':>12} {'sockets cliente':>15}")
for nombre, m in medidas.items():
    sockets_txt = "no disponible" if m["sockets"] is None else m["sockets"]
    print(f"{nombre:<13} {m['consultas']:>9} {m['demora']:>10.2f} {m['pendientes']:>21} "
          f"{m['sesiones_sse']:>12} {sockets_txt:>15}")

assert 1.5 <= medidas["polling"]["demora"] <= 3.0
assert medidas["long polling"]["demora"] < 0.5 and medidas["SSE"]["demora"] < 0.5
assert medidas["SSE"]["sse_al_final"] == 0, "quedaron sesiones SSE abiertas"

### Salida esperada de la celda 12.1

Una tabla con tres renglones. Los rangos son los que verifica la celda:

- **polling:** `consultas` = **80** (20 clientes × 4; 60 sin novedades), demora
  **entre 1.5 y 3.0 s** (esperado ~2 s), **0 long polls pendientes**, 0 sesiones SSE y
  **~20 sockets** del cliente (conexiones keep-alive abiertas entre
  consultas aunque no haya nada pendiente).
- **long polling:** `consultas` = **20** (una petición por cliente), demora
  **menos de 0.5 s**, **20 long polls pendientes** a mitad de la espera (el servidor
  retiene esas peticiones abiertas), 0 sesiones SSE.
- **SSE:** `consultas` = **20** (una apertura por cliente), demora **menos de
  0.5 s**, 0 long polls pendientes, **20 sesiones SSE** abiertas a mitad de la espera y 20 sockets; al
  final no queda ninguna sesión.

Si el polling sale con 79 u 81 consultas, un cliente cayó justo en el borde
del evento; repite la celda. Si `sockets cliente` dice «no disponible», tu
versión de httpx guarda el pool en otro lugar: no afecta el resto de la
medición. **Lectura.** Cambian tres cantidades distintas: cuántas peticiones
hace el cliente, cuánto tarda en enterarse y qué retiene el servidor
mientras espera. El mecanismo no decide por sí solo si se retiene un hilo o
una tarea.